# 1. Why does quantization matter in MoE? 

- MoE is already sparse, so fewer parameters are active -quantization should matter less than in dense models. 

- Partially true, but incomplete. In MoE, the bottleneck shifts from compute to memory. 

- Since each token can be routed to different experts scattered across GPUs, fetching their weights is expensive. 

- Quantization reduces weight size and therefore lowers than fetch cost. 

- The benefit comes not from compute savings but from data movement -an MoE -specific point that's often overlooked. 

# 2. Do experts become identical after quantization? 

- Exact equality is extremely unlikely -it would require millions of parameters to match after rounding.

- The real risk is functional similarity: quantization rounds away small differences, blurring the specialization boundaries between experts. 

- A nice analogy: pixelate two-high resolution images and they still look different, but the fine details are lost. 

# 3. The real danger: Expert Collapse

- The actual risk isn't experts "becoming the same" -it's some experts stopping being selected at all. Why? 

- Quantization add noise to expert outputs, which weakens the signals the router (gating network) receives. 

- Routing becomes less confident, some experts dominate, and others gradually go unused. 

- This explains why quantizing a MoE model sometimes causes unexpected benchmark degradation -not compute or memory, but the collapse of specialization. 

# 4. Don't quantize the router 

- The most practical takeaway: keep the router (gating network) in FP16 or FP32, and only quantize the expert weights. 

- The router is relatively small, so this costs almost nothing memory-wise, but it's critical for keeping routing stable. 